# Perbandingan KNN dan Decision Tree untuk Klasifikasi Presipitasi Harian di Medan

**Jenis tugas:** supervised learning, klasifikasi biner, perbandingan dua algoritma.

## Keputusan metodologi

- Unit observasi: satu hari.
- Fitur masukan: suhu rata-rata, kelembapan relatif rata-rata, tutupan awan rata-rata, tekanan permukaan laut rata-rata, dan kecepatan angin rata-rata pada hari yang sama.
- Target: `1` jika `precipitation_sum >= 10 mm`, `0` jika `precipitation_sum < 10 mm`.
- Ambang 10 mm diadopsi dari indeks klimatologi R10mm (hari dengan presipitasi harian minimal 10 mm). Dalam proyek ini, ambang tersebut digunakan untuk membentuk kelas biner, bukan untuk menghitung indeks tahunan R10mm.
- Model: K-Nearest Neighbors (KNN) dan Decision Tree.
- Evaluasi: Stratified 5-fold cross-validation, sesuai tugas perbandingan klasifikasi. Karena fitur dan target merupakan ringkasan dari hari yang sama, penelitian ini adalah **klasifikasi hari**, bukan prakiraan hujan untuk besok.

## Persiapan data

Sebelum menjalankan notebook, tempatkan file di:

`data/raw/open_meteo_medan_ecmwf_ifs_daily_2023_2025.csv`

CSV Open-Meteo dapat memuat dua baris metadata sebelum header tabel, jadi notebook membacanya dengan `skiprows=2`.

**Penting tentang sumber model:** CSV yang sebelumnya dikirim tidak mencatat model reanalysis di dalam file. Untuk hasil yang bisa direproduksi, unduh ulang dari Historical Weather API dengan mencentang **ECMWF IFS saja** (jangan memilih Best Match) dan variabel Daily yang sama. Catat pengaturan sumber dalam `data/SOURCE.md`. Dokumentasi: https://open-meteo.com/en/docs/historical-weather-api

## Referensi metodologis utama

1. Oliveira et al. (2023), *Precipitation forecasting: from geophysical aspects to machine learning applications*, Frontiers in Climate. Tinjauan ini menjelaskan pipeline ML berupa pengumpulan data, pembersihan/preprocessing, feature engineering, training, dan testing/validation. https://doi.org/10.3389/fclim.2023.1250201
2. Wani et al. (2024), *Predicting rainfall using machine learning, deep learning, and time series models across an altitudinal gradient in the North-Western Himalayas*, Scientific Reports. Studi menggunakan data meteorologis harian, melakukan preprocessing, membagi data untuk training/testing, dan menilai model dengan metrik yang sesuai jenis tugasnya. https://doi.org/10.1038/s41598-024-77687-x
3. Birant et al. (2025), *A new machine learning method for rainfall classification: temporal random tree*, PeerJ Computer Science. Studi ini membahas klasifikasi rain/no-rain dan pentingnya aspek temporal pada data presipitasi. https://doi.org/10.7717/peerj-cs.3022
4. Definisi indeks R10mm: Climdex, https://www.climdex.org/learn/indices/
5. Dokumentasi scikit-learn tentang pipeline dan cross-validation: https://scikit-learn.org/stable/modules/cross_validation.html dan https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html

**Catatan keterbatasan:** fitur cuaca dan label diambil dari sumber historis model yang sama. Hasil menilai kemampuan mengklasifikasikan kategori harian pada dataset ini, bukan akurasi terhadap observasi stasiun independen atau prediksi operasional masa depan.

## 1. Import library dan siapkan direktori keluaran

In [2]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

# Temukan root repository jika notebook dijalankan dari folder notebooks/ atau root.
current = Path.cwd().resolve()
ROOT = next(
    (p for p in [current, *current.parents]
     if (p / "data" / "raw").exists() and (p / "notebooks").exists()),
    current
)

RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
FIGURES_DIR = ROOT / "reports" / "figures"
TABLES_DIR = ROOT / "reports" / "tables"

for folder in [PROCESSED_DIR, FIGURES_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

DATA_PATH = RAW_DIR / "open_meteo_medan_ecmwf_ifs_daily_2023_2025.csv"
print("Repository root:", ROOT)
print("Input CSV:", DATA_PATH)
print("File exists:", DATA_PATH.exists())

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "CSV tidak ditemukan. Simpan CSV di data/raw/ dengan nama "
        "open_meteo_medan_ecmwf_ifs_daily_2023_2025.csv"
    )

Repository root: E:\SEMESTER 5 ILMU KOMPUTER USU\Kecerdasan Buatan\TUGAS prof Poltak\Tugas Pertemuan 9
Input CSV: E:\SEMESTER 5 ILMU KOMPUTER USU\Kecerdasan Buatan\TUGAS prof Poltak\Tugas Pertemuan 9\data\raw\open_meteo_medan_ecmwf_ifs_daily_2023_2025.csv
File exists: True


## 2. Baca CSV dan rapikan nama kolom

In [3]:
# Dua baris pertama export Open-Meteo menyimpan metadata lokasi.
df = pd.read_csv(DATA_PATH, skiprows=2)

# Hilangkan satuan yang ditulis dalam tanda kurung di header CSV.
df.columns = [str(col).split(" (")[0].strip() for col in df.columns]
df = df.rename(columns={"time": "date"})

required_cols = [
    "date",
    "temperature_2m_mean",
    "relative_humidity_2m_mean",
    "cloud_cover_mean",
    "pressure_msl_mean",
    "wind_speed_10m_mean",
    "precipitation_sum",
    "rain_sum",
]
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(
        f"Kolom berikut tidak ditemukan: {missing_cols}. "
        f"Kolom yang terbaca: {df.columns.tolist()}"
    )

df["date"] = pd.to_datetime(df["date"], errors="coerce")
numeric_cols = [col for col in required_cols if col != "date"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Kolom yang terbaca:")
print(df.columns.tolist())
display(df.head())

Kolom yang terbaca:
['date', 'temperature_2m_mean', 'relative_humidity_2m_mean', 'cloud_cover_mean', 'pressure_msl_mean', 'wind_speed_10m_mean', 'precipitation_sum', 'rain_sum']


,date,temperature_2m_mean,relative_humidity_2m_mean,cloud_cover_mean,pressure_msl_mean,wind_speed_10m_mean,precipitation_sum,rain_sum
0,2023-01-01,25.0,84,100,1012.6,8.3,1.1,1.1
1,2023-01-02,24.5,89,98,1013.0,7.8,17.8,17.8
2,2023-01-03,24.7,89,100,1012.6,6.4,5.8,5.8
3,2023-01-04,25.2,89,100,1012.0,6.3,7.4,7.4
4,2023-01-05,25.0,90,100,1011.8,8.8,5.0,5.0
